### Installing Required Libraries

The next cell installs the necessary libraries, `dask` and `pandas`. The `dask[complete]` package is used for distributed computing and handling large datasets, while `pandas` is a library for data manipulation and analysis. This ensures that all required dependencies are available for the subsequent operations in the notebook.

In [1]:
%pip install "dask[complete]" pandas


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

# Size parameters (Adjust to be larger than your memory)
# 10M rows * 10 columns = 100M * 8 bytes per float = ~760MB 
num_rows = 10**7  # 10 million rows
num_cols = 10     # 10 columns
# Save the DataFrame to a CSV file
csv_filename = 'data/large_data.csv'


### Generating a Large Dataset

The next cell generates a large synthetic dataset using NumPy and Pandas. It creates a DataFrame with 10 million rows and 10 columns filled with random numbers. The dataset is then saved as a CSV file named `large_data.csv` in the `data` directory. This large dataset will be used to demonstrate the capabilities of both Pandas and Dask in handling and processing large-scale data.

In [3]:

# Generate random data
df_large = pd.DataFrame(np.random.randn(num_rows, num_cols), columns=[f'col{i}' for i in range(num_cols)])

df_large.to_csv(csv_filename, index=False)

### Loading the Dataset with Pandas

The next cell demonstrates how to load the previously generated large dataset using Pandas. It reads the `large_data.csv` file into a Pandas DataFrame and displays information about the dataset, such as the number of rows, columns, and data types.

In [4]:
df_pandas = pd.read_csv(csv_filename)
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 10 columns):
 #   Column  Dtype  
---  ------  -----  
 0   col0    float64
 1   col1    float64
 2   col2    float64
 3   col3    float64
 4   col4    float64
 5   col5    float64
 6   col6    float64
 7   col7    float64
 8   col8    float64
 9   col9    float64
dtypes: float64(10)
memory usage: 762.9 MB


### Processing the Dataset with Pandas

The next cell attempts to process the large dataset using Pandas. It performs a simple operation of computing the mean of each column in the DataFrame. If the dataset is too large to fit into memory, a `MemoryError` will be raised, and an appropriate message will be displayed. This demonstrates the limitations of Pandas when handling datasets that exceed the available memory.

In [5]:
%%time
# Attempt to load and process with pandas
try:
    # Perform a simple operation like computing the mean of each column
    mean_values_pandas = df_pandas.mean()
    print("Pandas successfully processed the dataframe:")
    print(mean_values_pandas)
except MemoryError:
    print("Pandas failed due to a memory error.")

Pandas successfully processed the dataframe:
col0    0.000116
col1    0.000056
col2   -0.000482
col3   -0.000281
col4    0.000653
col5    0.000074
col6    0.000192
col7    0.000147
col8   -0.000055
col9    0.000863
dtype: float64
CPU times: user 122 ms, sys: 11.2 ms, total: 133 ms
Wall time: 127 ms


### Loading and Processing the Dataset with Dask

The next cell demonstrates how to load and process the large dataset using Dask. Unlike Pandas, Dask can handle datasets that exceed the available memory by splitting the data into smaller partitions and processing them in parallel. The cell first loads the `large_data.csv` file into a Dask DataFrame and then calculates the number of partitions and the total memory usage of the dataset. 

In [6]:
import dask.dataframe as dd

# Load and process with Dask
df_dask = dd.read_csv(csv_filename)

np = df_dask.npartitions
mem_usage = df_dask.memory_usage_per_partition().sum().compute()

print(f"This Dask dataframe has {np} partitions")
print(f"Total memory consumption is {mem_usage/1024/1024} MB")


This Dask dataframe has 30 partitions
Total memory consumption is 762.943229675293 MB


### Computing the Mean with Dask

The next cell demonstrates how to compute the mean of each column in the large dataset using Dask. Unlike Pandas, Dask processes the data in parallel across multiple partitions, allowing it to handle datasets that exceed the available memory. The `mean()` method is used to calculate the mean for each column, and the `compute()` method triggers the computation. 

In [7]:
%%time
# Perform the same operation
mean_values_dask = df_dask.mean().compute()
print("Dask successfully processed the dataframe:")
print(mean_values_dask)

Dask successfully processed the dataframe:
col0    0.000116
col1    0.000056
col2   -0.000482
col3   -0.000281
col4    0.000653
col5    0.000074
col6    0.000192
col7    0.000147
col8   -0.000055
col9    0.000863
dtype: float64
CPU times: user 12 s, sys: 2.63 s, total: 14.6 s
Wall time: 2.62 s
